# Definizioni iniziali

### Pacchetti

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Matplotlib

In [ ]:
def show_image(img, title=None, cmap=None):
    cmap = cmap or ('gray' if len(img.shape) == 2 else None)
    img = img
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    figure, axes = plt.subplots(figsize=(20, 20))
    axes.imshow(img, cmap=cmap, vmin=0)
    axes.set_title(title)
    axes.axis('off')
    plt.show()

def show_images(images, cmap=None):
    # Se l'immagine è un array monodimensionale, la trasformo in un array bidimensionale
    if isinstance(images[0], tuple):
        images = [images]
    figure, subplots = plt.subplots(len(images), len(images[0]), figsize=(20, 20))
    figure.tight_layout()
    for row in range(len(images)):
        for column in range(len(images[row])):
            # Se l'immagine è un array di tre elementi, allora il terzo elemento è il cmap
            if len(images[row][column]) == 3:
                img, title, _cmap = images[row][column]
            else:
                img, title = images[row][column]
                _cmap = cmap or ('gray' if len(img.shape) == 2 else None)

            # Se l'immagine è a colori, la converto in RGB per poterla visualizzare correttamente
            if len(img.shape) == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Se subplots è un array monodimensionale, allora non c'è bisogno di usare la notazione a due dimensioni
            if len(images) == 1:
                subplots[column].imshow(img, cmap=_cmap, vmin=0)
                subplots[column].set_title(title)
                subplots[column].axis('off')
            else:
                subplots[row, column].imshow(img, cmap=_cmap, vmin=0)
                subplots[row, column].set_title(title)
                subplots[row, column].axis('off')
    plt.show()

# Maschera

#### Binarizzazione

Viene binarizzata l'immagine con una soglia di 230, che ha dato buoni risultati, per dividere le parti di sfondo della foto dal sample vero.

**Lo sfondo diventerà bianco nella binarizzazione, mentre il sample diventerà nero. OpenCV chiama inversamente queste due parti: lo sfondo nero e il foreground bianco.**

In [ ]:
label_free = cv2.imread("../../Materiale/Locale/cut_images/alto_sx_1.png")

# Binarizza l'immagine con una soglia
_, binary = cv2.threshold(cv2.cvtColor(label_free, cv2.COLOR_BGR2GRAY), 230, 255, cv2.THRESH_BINARY)

show_images([(label_free, "Non colorata"), (binary, "Binaria")])

#### Rilevamento dei componenti
`cv2.connectedComponentsWithStats` rileva tutti i componenti connessi (blob) nell'immagina binarizzata. Restituisce una immagine delle stesse dimensioni dell'immagine iniziale, assegnando a ogni pixel appartente a un blob il medesimo valore. In questo modo, tutti i pixel del blob 13 avranno valore 13. I pixel di valore 0 appartengono allo sfondo, che saltiamo.

I componenti vengono ordinati per area, dato che è più probabile che lo sfondo sia abbastanza grande rispetto all'immagine. Essendo più grande, è anche più rilevante ai fini della maschera.

In [ ]:
# Trova i componenti connessi
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
print(f"Number of components: {num_labels}")

# Ordina in modo decrescente i componenti per area
n_filtered = 100
sorted_indices = np.argsort(stats[1:, cv2.CC_STAT_AREA])[::-1] + 1
filtered_labels = np.zeros_like(labels)
for i in sorted_indices[1:n_filtered]:
    filtered_labels[labels == i] = i
show_images([(labels, "Etichette"), (filtered_labels, "Etichette filtrate")], cmap='nipy_spectral')

#### Filtraggio per deviazione standard

Il calcolo della deviazione standard σ permette di vedere quanto variano di intensità i pixel in una certa area. \
Per ogni componente ne si ottiene la zona di interesse (ovvero l'area occupata dal componente). Successivamente si ottiene la maschera del componente, che ci indica quale è effettivamente la forma del componente, e la si riempire, rimuovendo eventuali punti di non-maschera all'interno del componente. Questo è necessario perché poi si calcola la deviazione standard dell'immagine originale nei punti dove si trova il componente. \
Se il componente non è lo sfondo, ci sarà molta variazione di intensità dei pixel, e di conseguenza un'elevata deviazione standard, che filtriamo a > 10.

Riempire la maschera del componente è necessario altrimenti il calcolo della deviazione standard ignorerebbe i punti molto diversi, rovinandone il significato. Il riempimento però comporta il rischio nel caso in cui vi si trovi "un'isola" vera di sample in uno sfondo, poiché in tal caso lo sfondo verrà filtrato avendo deviazione standard elevata. \
Per questo conviene calcolare la maschera dell'intera immagine a più livelli di grandezza e poi unire tutte le maschere per ottenere una maschera finale.

In [ ]:
# Crea una maschera vuota per filtrare i componenti
mask_label_free = np.zeros_like(binary)

images = []
# Per ogni componente in ordine decrescente di area
for i in sorted_indices[:n_filtered]:
    x, y, w, h, area = stats[i]

    # Riempe la maschera
    component_mask = (labels == i).astype(np.uint8) * 255
    countours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(component_mask, countours, -1, 255, thickness=cv2.FILLED)
    
    # Estrae dalla regione di interesse
    roi = label_free[y:y+h, x:x+w]
    roi_mask = component_mask[y:y+h, x:x+w]

    # Calcola la deviazione standard della regione di interesse
    std_dev = cv2.meanStdDev(roi, mask=roi_mask)[1][0, 0]
    images.append([(roi, f"Componente {i}"), (roi_mask, f"Maschera σ {std_dev:.2f}")])

    # Filtra i componenti con una deviazione standard troppo alta
    if std_dev < 10:
        mask_label_free[labels == i] = 255
show_images(images[:10])

#### Applicazione della maschera 

In [ ]:
# Applica la maschera all'immagine
mask_label_free = cv2.bitwise_not(mask_label_free)
masked_label_free = cv2.bitwise_and(label_free, label_free, mask=mask_label_free)

# Mostra le immagini
show_images([
    (label_free, "Non colorata"),
    (mask_label_free, "Maschera"),
    (masked_label_free, "Non colorata con maschera")])